In [ ]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as tt

import graphviz

In [ ]:
muts = pd.read_csv('annot_muts_snp_m2indels.csv')
muts.head()

In [ ]:
n_muts_syn = (
    muts
    .query('impact == "Synonymous"')
    .groupby(['sampleID','gene'])
    .pid.count()
    .reset_index()
    .pivot(index='gene',columns='sampleID',values='pid')
    .fillna(0)
)

n_muts_nsyn = (
    muts
    .query('impact != "Synonymous"')
    .groupby(['sampleID','gene'])
    .pid.count()
    .reset_index()
    .pivot(index='gene',columns='sampleID',values='pid')
    .fillna(0)
)

n_muts_syn

In [ ]:
n_syn = (n_muts_syn.sum(axis=1)+n_muts_nsyn.sum(axis=1)-n_muts_syn.sum(axis=1)).fillna(0)
tot_mut = (n_muts_syn.sum(axis=1)+n_syn).fillna(0)

In [ ]:
tot_mut

In [ ]:
n_syn_vec = n_syn[n_syn!=0]
tot_mut_vec = tot_mut[n_syn!=0]

In [ ]:
gene_names = tot_mut_vec.sort_values().sample(20, random_state=1).index

In [ ]:
n_syn_vec = n_syn_vec.loc[gene_names].to_numpy()
tot_mut_vec = tot_mut_vec.loc[gene_names].to_numpy()

In [ ]:
tot_mut_vec

## Unpooled

In [ ]:
N = len(n_syn_vec)

with pm.Model() as unpooled:
    thetas = pm.Beta("thetas", alpha=1, beta=1, shape=N)
    y = pm.Binomial("y", n=tot_mut_vec, p=thetas, observed=n_syn_vec)

In [ ]:
pm.model_to_graphviz(unpooled)

In [ ]:
with unpooled:
    trace_unpooled = pm.sample(2000, tune=200, chains=2, target_accept=0.80, return_inferencedata=True)

    # check convergence diagnostics
    assert all(az.rhat(trace_unpooled) < 1.03)

In [ ]:
ax = az.plot_forest(trace_unpooled, var_names=["thetas"])
ax[0].set_yticklabels(gene_names.tolist());

### Partially pooled

In [ ]:
N = len(n_syn_vec)

with pm.Model() as partial_pool:

    phi = pm.Gamma("phi", alpha=1, beta=1)
    kappa = pm.Gamma("kappa", alpha=1, beta=1)

    thetas = pm.Beta("thetas", alpha=phi, beta=kappa, shape=N)
    y = pm.Binomial("y", n=tot_mut_vec, p=thetas, observed=n_syn_vec)

In [ ]:
pm.model_to_graphviz(partial_pool)

In [ ]:
with partial_pool:
    trace = pm.sample(2000, tune=200, chains=2, target_accept=0.80, return_inferencedata=True)

    # check convergence diagnostics
    assert all(az.rhat(trace) < 1.03)

In [ ]:
az.plot_trace(trace, var_names=["phi", "kappa"]);

## Pooled

In [ ]:
N = len(n_syn_vec)

with pm.Model() as pooled:

    theta = pm.Beta("thetas", alpha=1, beta=1, shape=1)
    y = pm.Binomial("y", n=tot_mut_vec, p=theta, observed=n_syn_vec)

In [ ]:
pm.model_to_graphviz(pooled)

In [ ]:
with pooled:
    trace_pooled = pm.sample(2000, tune=200, chains=2, target_accept=0.80, return_inferencedata=True)

    # check convergence diagnostics
    assert all(az.rhat(trace) < 1.03)

In [ ]:
az.plot_trace(trace_pooled, var_names=["thetas"]);

## Comparison

In [ ]:
ax = az.plot_forest([trace_pooled,trace_unpooled,trace],
                    model_names=['Pooled', 'Unpooled', 'Partial'], var_names=["thetas"])
ax[0].set_yticklabels(gene_names.tolist())
ax[0].vlines(x=0.5, ymin=0, ymax=600, color='r', linestyles='--')